# ✍️ Session 3: Prompt Engineering Fundamentals
### Agentic AI Nano Bootcamp | Day 1 · Session 3

---

## 🎯 Learning Objectives
By the end of this session you will be able to:
- Describe the elements of an effective prompt
- Apply zero-shot, few-shot, chain-of-thought, and instructional prompting
- Use prompt templates for consistent, reusable outputs
- Control output format, tone, and length through prompts
- Optimize chatbot responses using prompt engineering techniques

## 📋 Session Outline
1. What is Prompt Engineering?
2. Elements of a Good Prompt
3. Prompt Types with Examples
4. Controlling Output Format
5. Prompt Templates
6. 🔬 Lab: Prompt Type Experiments (One per type)
7. 🔬 Lab: Chatbot Response Optimization

---

In [ ]:
# ── Setup ──
import os, subprocess
subprocess.run(['pip', 'install', 'openai', 'python-dotenv', '-q'], capture_output=True)

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
client = OpenAI()

def prompt(user_msg, system_msg="You are a helpful assistant.", temperature=0.7, max_tokens=400, model="gpt-4o-mini"):
    """Convenience wrapper — returns response text."""
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return r.choices[0].message.content

def compare_prompts(label_a, prompt_a, label_b, prompt_b, system="You are a helpful assistant."):
    """Side-by-side comparison of two prompts."""
    print(f'\n{"="*60}')
    print(f'  Prompt A — {label_a}')
    print(f'{"="*60}')
    print(f'📤 Prompt: {prompt_a[:100]}...' if len(prompt_a)>100 else f'📤 Prompt: {prompt_a}')
    print(f'📥 Output:\n{prompt(prompt_a, system)}')
    print(f'\n{"="*60}')
    print(f'  Prompt B — {label_b}')
    print(f'{"="*60}')
    print(f'📤 Prompt: {prompt_b[:100]}...' if len(prompt_b)>100 else f'📤 Prompt: {prompt_b}')
    print(f'📥 Output:\n{prompt(prompt_b, system)}')

print('✅ Setup complete — client ready')

## 1. What is Prompt Engineering?

**Prompt engineering** is the practice of designing and refining natural language inputs to reliably get high-quality, useful outputs from a language model.

It sits at the intersection of **art and science**: part precise instruction-writing, part experimentation, part psychology of communication.

### Why It Matters

| Poor Prompt | Engineered Prompt | Difference |
|---|---|---|
| "summarize this" | "Summarize in 3 bullet points for a non-technical executive audience" | Audience + format + length |
| "write code" | "Write a Python function with type hints, docstring, and error handling" | Specificity |
| "is this good?" | "Rate this ticket response on clarity (1-5) and empathy (1-5). Explain each score." | Structured criteria |

### The Prompt Engineering Stack
```
┌─────────────────────────────────────────┐
│     Prompt Templates (reusable)         │  ← Level 4
├─────────────────────────────────────────┤
│     Advanced: CoT, ReAct, Self-Ask      │  ← Level 3
├─────────────────────────────────────────┤
│     Few-shot Examples                   │  ← Level 2
├─────────────────────────────────────────┤
│     Zero-shot + Role + Format           │  ← Level 1
└─────────────────────────────────────────┘
```

## 2. Elements of a Good Prompt

A high-quality prompt typically contains some or all of these elements:

```
┌──────────────┬──────────────────────────────────────────────────────┐
│ Element      │ Purpose                                              │
├──────────────┼──────────────────────────────────────────────────────┤
│ ROLE         │ Tell the model who it is                             │
│              │ "You are a senior telecom network engineer..."        │
├──────────────┼──────────────────────────────────────────────────────┤
│ CONTEXT      │ Background info the model needs                      │
│              │ "The customer has a 100 Mbps fiber plan..."           │
├──────────────┼──────────────────────────────────────────────────────┤
│ TASK         │ Exactly what to do                                   │
│              │ "Write a 3-step troubleshooting guide..."             │
├──────────────┼──────────────────────────────────────────────────────┤
│ FORMAT       │ How to structure the output                          │
│              │ "Respond as JSON / bullet points / markdown table"    │
├──────────────┼──────────────────────────────────────────────────────┤
│ EXAMPLES     │ Show 1–3 input→output pairs (few-shot)               │
│              │ "Input: X → Output: Y"                               │
├──────────────┼──────────────────────────────────────────────────────┤
│ CONSTRAINTS  │ What NOT to do / length / audience                   │
│              │ "Keep it under 100 words. No jargon."                 │
└──────────────┴──────────────────────────────────────────────────────┘
```

In [ ]:
# ── Demo: Elements of a Prompt — Adding them One by One ──

complaint = "My internet is very slow and I want a refund."

# Level 1: Just the task
p1 = f"Respond to this customer complaint: {complaint}"

# Level 2: Task + Role
p2 = f"""You are a senior customer service agent at a telecom company.
Respond to this customer complaint: {complaint}"""

# Level 3: Task + Role + Format
p3 = f"""You are a senior customer service agent at a telecom company.
Respond to this customer complaint: {complaint}

Format:
1. Empathetic opening (1 sentence)
2. What you will do (2 steps)
3. Closing with timeline"""

# Level 4: Task + Role + Format + Constraints
p4 = f"""You are a senior customer service agent at a telecom company.
Respond to this customer complaint: {complaint}

Format:
1. Empathetic opening (1 sentence)
2. What you will do (2 steps)
3. Closing with timeline

Constraints:
- Maximum 80 words total
- Avoid technical jargon
- Do not promise a refund directly — offer to investigate
- Tone: warm and professional"""

levels = [
    ('Level 1: Task only',              p1),
    ('Level 2: + Role',                 p2),
    ('Level 3: + Format',               p3),
    ('Level 4: + Constraints',          p4),
]

for label, p in levels:
    print(f'\n🔧 {label}:')
    print('-' * 50)
    print(prompt(p, temperature=0.3))

## 3. Prompt Types

### Overview of Prompting Techniques

| Technique | When to Use | How |
|---|---|---|
| **Zero-shot** | Simple, clear tasks | Just describe the task |
| **Few-shot** | Pattern-following tasks | Provide 2–5 examples |
| **Chain-of-Thought** | Complex reasoning | Ask model to "think step by step" |
| **Role prompting** | Domain expertise needed | Assign a professional persona |
| **Instructional** | Precise formatted output | Give explicit instructions + format |
| **Self-consistency** | Critical decisions | Generate multiple answers, pick best |
| **Negative prompting** | Avoid bad outputs | Explicitly state what NOT to do |

In [ ]:
# ── Prompt Type 1: Zero-Shot ──
# No examples provided — model uses its training knowledge

print('🎯 ZERO-SHOT PROMPTING')
print('=' * 60)
print('Concept: Give a task description with NO examples.')
print('Best for: Well-defined tasks the model knows well.\n')

zero_shot_examples = [
    (
        "Classification",
        """Classify the following telecom support ticket into one of these categories:
        [Billing, Connectivity, Hardware, Account, Upgrade]
        
        Ticket: 'My router's WiFi light is blinking red and I have no internet.'
        
        Respond with only the category name."""
    ),
    (
        "Sentiment",
        """What is the sentiment of this review? Answer with one word: Positive, Negative, or Neutral.
        
        Review: 'The speed is decent but the price is way too high for what you get.'"""
    ),
    (
        "Extraction",
        """Extract the following fields from the customer report and return as JSON:
        - customer_id
        - issue_type  
        - duration_hours
        - affected_services
        
        Report: 'Customer ID TN-9182 reports complete loss of broadband and TV for 6 hours 
        since this morning. Account verified active.'"""
    )
]

for task, p in zero_shot_examples:
    print(f'\n📌 Zero-shot {task}:')
    print(f'Prompt preview: {p[:80].strip()}...')
    print(f'Output: {prompt(p, temperature=0.0)}')

In [ ]:
# ── Prompt Type 2: Few-Shot ──
# Provide 2-5 examples to teach the model the exact pattern you want

print('🎯 FEW-SHOT PROMPTING')
print('=' * 60)
print('Concept: Show the model INPUT → OUTPUT examples before the real task.')
print('Best for: Custom formats, domain-specific patterns, classification.\n')

# Use case: Ticket urgency scoring in a specific format
few_shot_prompt = """
You score support tickets by urgency. Use format: SCORE: X/5 | REASON: <brief>

Examples:
Ticket: "My bill seems a bit high this month."
SCORE: 1/5 | REASON: Billing query, non-urgent, can wait for next billing cycle

Ticket: "Internet down for 2 days, work from home, can't attend meetings."
SCORE: 4/5 | REASON: Productivity impact, multi-day outage, needs same-day resolution

Ticket: "Hospital ICU monitoring system lost connectivity. Patient data not syncing."
SCORE: 5/5 | REASON: Critical safety risk, medical infrastructure, immediate escalation required

Now score these tickets:
"""

test_tickets = [
    "My Netflix is buffering. The speed test shows 15 Mbps but I pay for 100 Mbps.",
    "ATM network down at our bank branch. Customers cannot withdraw cash.",
    "WiFi password changed and I can't remember the new one.",
]

for ticket in test_tickets:
    full_prompt = few_shot_prompt + f'Ticket: "{ticket}"\n'
    result = prompt(full_prompt, temperature=0.1, max_tokens=80)
    print(f'Ticket: {ticket[:60]}...')
    print(f'  → {result}\n')

In [ ]:
# ── Prompt Type 3: Chain-of-Thought (CoT) ──
# Ask the model to reason step-by-step before giving an answer

print('🎯 CHAIN-OF-THOUGHT PROMPTING')
print('=' * 60)
print('Concept: Ask model to "think step by step" before answering.')
print('Best for: Math, reasoning, multi-step decisions, debugging.\n')

problem = """
A telecom tower serves 3,000 subscribers.
Currently 60% are on 4G (8 Mbps average) and 40% are on 5G (80 Mbps average).
The tower has 12 Gbps total capacity.
Is the tower over capacity? By how much (in Gbps)?
What percentage of capacity is being used?
"""

# Without CoT
plain_answer = prompt(
    f"Answer this question:\n{problem}",
    temperature=0.0
)

# With CoT
cot_answer = prompt(
    f"Think step by step and show your calculations before giving the final answer:\n{problem}",
    temperature=0.0
)

print('Without CoT:')
print('-' * 40)
print(plain_answer)
print('\nWith Chain-of-Thought:')
print('-' * 40)
print(cot_answer)

In [ ]:
# ── Prompt Type 4: Role Prompting ──
# Assign an expert persona to get domain-specific, authoritative responses

print('🎯 ROLE PROMPTING')
print('=' * 60)
print('Concept: Assign a professional persona to shift knowledge domain + tone.')
print('Best for: Domain-specific advice, tonal consistency, audience targeting.\n')

same_question = "Our network latency jumped from 20ms to 180ms after the new router config. What happened?"

roles = [
    "You are a Level 1 helpdesk agent. Give a simple explanation for a non-technical customer.",
    "You are a senior network engineer with 20 years of experience. Give a detailed technical RCA.",
    "You are a CTO preparing a 2-sentence executive briefing for the board."
]

for role in roles:
    print(f'\n👤 Role: {role[:80]}')
    print('-' * 50)
    print(prompt(same_question, system_msg=role, temperature=0.4))

In [ ]:
# ── Prompt Type 5: Instructional Prompting ──
# Very precise, structured instructions for predictable, parseable outputs

print('🎯 INSTRUCTIONAL PROMPTING')
print('=' * 60)
print('Concept: Specify exact output structure using headers, JSON, or schemas.')
print('Best for: Automation pipelines, structured data extraction, APIs.\n')

ticket_text = """
Customer Rajesh Kumar (Account: TN-291847) called at 3:15 PM today. 
He is experiencing complete loss of broadband since 8 AM. He's on the 
Gold 200 Mbps plan. Router lights show solid green for power but blinking 
red for internet. He already tried power cycling twice. He works from home 
and has missed two important meetings. He is very frustrated and requesting 
compensation. His contact: 9884XXXXXX.
"""

instructional_prompt = f"""
Extract information from the support call transcript and return ONLY valid JSON.
No explanations, no markdown, no preamble.

Required JSON structure:
{{
  "customer": {{
    "name": string,
    "account_id": string,
    "contact": string,
    "plan": string
  }},
  "incident": {{
    "type": string,
    "start_time": string,
    "duration_estimate": string,
    "symptoms": [list of strings],
    "steps_taken": [list of strings]
  }},
  "customer_impact": {{
    "severity": "Low|Medium|High|Critical",
    "business_impact": boolean,
    "compensation_requested": boolean,
    "emotion": string
  }},
  "recommended_priority": "P1|P2|P3|P4"
}}

Transcript:
{ticket_text}
"""

result = prompt(instructional_prompt, temperature=0.0)
print('📋 Structured JSON Output:')
print(result)

# Parse and use the result
import json
try:
    parsed = json.loads(result)
    print(f'\n✅ Valid JSON! Priority: {parsed["recommended_priority"]}')
    print(f'   Customer: {parsed["customer"]["name"]}')
    print(f'   Severity: {parsed["customer_impact"]["severity"]}')
except json.JSONDecodeError as e:
    print(f'⚠️  JSON parse error: {e}')

In [ ]:
# ── Prompt Type 6: Negative Prompting ──
# Explicitly tell the model what NOT to do

print('🎯 NEGATIVE PROMPTING (Do Not / Avoid / Never)')
print('=' * 60)
print('Concept: Explicitly exclude unwanted behaviors or content.\n')

compare_prompts(
    label_a="Without negative prompts",
    prompt_a="Write a response to a customer who says their internet is slow.",
    label_b="With negative prompts",
    prompt_b="""Write a response to a customer who says their internet is slow.
    
    DO NOT:
    - Promise a specific resolution timeline
    - Mention technical terms like 'bandwidth', 'latency', 'DNS'
    - Offer a discount or refund
    - Use phrases like 'I apologize for the inconvenience'
    
    DO:
    - Show genuine empathy
    - Ask one clarifying question
    - Keep under 50 words"""
)

## 4. Controlling Output Format

You can instruct the model to return output in any format: plain text, JSON, Markdown, CSV, tables, code, YAML, or custom schemas. This is critical for production systems that parse the output programmatically.

In [ ]:
# ── Output Format Control ──

topic = "Top 3 differences between 4G and 5G networks"

formats = {
    'Plain text':      f"Explain: {topic}",
    'Bullet points':   f"List as bullet points (•): {topic}",
    'Numbered list':   f"List as numbered steps: {topic}",
    'Markdown table':  f"Create a markdown comparison table: {topic}",
    'JSON array':      f"Return as a JSON array of objects with fields: 'aspect', '4G', '5G'. No extra text. {topic}",
    'Tweet-length':    f"In exactly one tweet (under 280 characters): {topic}",
}

for fmt, p in formats.items():
    print(f'\n📐 Format: {fmt}')
    print('-' * 40)
    print(prompt(p, temperature=0.3, max_tokens=200))

## 5. Prompt Templates

Prompt templates are **reusable prompt structures with variable placeholders**. They ensure consistency across an application and make it easy to swap out content without rewriting the logic.

```python
# Simple template
TEMPLATE = """
You are a {role}.
Task: {task}
Context: {context}
Format: {format}
"""
```

In [ ]:
# ── Prompt Template Library ──

class PromptTemplate:
    """A reusable prompt template with variable substitution."""
    
    def __init__(self, template: str, required_vars: list = None):
        self.template = template
        self.required_vars = required_vars or []
    
    def format(self, **kwargs) -> str:
        missing = [v for v in self.required_vars if v not in kwargs]
        if missing:
            raise ValueError(f'Missing required variables: {missing}')
        return self.template.format(**kwargs)
    
    def run(self, client, temperature=0.3, max_tokens=400, **kwargs) -> str:
        filled = self.format(**kwargs)
        r = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': filled}],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return r.choices[0].message.content


# ── Template Library: Telecom Use Cases ──

TICKET_CLASSIFIER = PromptTemplate(
    template="""Classify this support ticket into one category: {categories}
    Also give a confidence score (0-100) and a one-line reason.
    Format: CATEGORY | CONFIDENCE | REASON
    
    Ticket: {ticket}""",
    required_vars=['ticket', 'categories']
)

RESPONSE_DRAFTER = PromptTemplate(
    template="""You are a {role} at a telecom company.
    Draft a response to the customer using the following information:
    Issue: {issue}
    Resolution: {resolution}
    Tone: {tone}
    Max words: {max_words}""",
    required_vars=['role', 'issue', 'resolution', 'tone', 'max_words']
)

REPORT_GENERATOR = PromptTemplate(
    template="""Generate a {report_type} report for the following incident.
    Include: executive summary, root cause, impact, resolution steps, and preventive measures.
    Audience: {audience}
    Format: Markdown with headers.
    
    Incident: {incident_details}""",
    required_vars=['report_type', 'audience', 'incident_details']
)

# ── Demo: Using the Templates ──

print('📋 Template 1: Ticket Classifier')
print('-' * 50)
result = TICKET_CLASSIFIER.run(
    client,
    ticket="My broadband speed is 5 Mbps but I pay for 100 Mbps.",
    categories="Billing, Connectivity, Hardware, Account, Upgrade, Other"
)
print(result)

print('\n📋 Template 2: Response Drafter')
print('-' * 50)
result = RESPONSE_DRAFTER.run(
    client,
    role="senior customer success manager",
    issue="Complete internet outage for 4 hours",
    resolution="Issue identified as fiber cut, repaired, service restored",
    tone="empathetic and professional",
    max_words="75"
)
print(result)

## 🔬 Lab: Chatbot Response Optimization

In this lab we will take a **poorly-designed chatbot** and progressively improve it using prompt engineering techniques until it performs at a professional level.

### The Scenario
A telecom company wants a customer support chatbot that:
- Handles common queries (billing, connectivity, plans)
- Responds with empathy and professionalism
- Follows a consistent format
- Escalates when appropriate

In [ ]:
# ── Step 1: The Naive Chatbot (baseline) ──

def naive_chatbot(user_query):
    return prompt(user_query)

test_queries = [
    "Why is my bill ₹200 more than last month?",
    "My internet is down and I have an important presentation in 30 minutes!",
    "I want to cancel my subscription."
]

print('❌ NAIVE CHATBOT (no engineering):')
print('=' * 60)
for q in test_queries:
    print(f'\n👤 {q}')
    print(f'🤖 {naive_chatbot(q)[:200]}...' if len(naive_chatbot(q)) > 200 else f'🤖 {naive_chatbot(q)}')

In [ ]:
# ── Step 2: The Optimized Chatbot ──

OPTIMIZED_SYSTEM_PROMPT = """
You are Aria, a senior customer support specialist at SwiftNet Telecom.
You have 5 years of experience and are known for resolving issues on first contact.

RESPONSE STRUCTURE (always follow this):
1. Empathy line (1 sentence acknowledging the customer's feeling)
2. Diagnosis or explanation (1-2 sentences, plain language)
3. Action steps (numbered, max 3 steps)
4. Closing (reassurance + contact option)

RULES:
- Always address the customer as 'you', never 'the customer'
- No technical jargon without explanation
- For billing: offer to review within 24 hours, never confirm errors immediately
- For cancellations: acknowledge, ask one question about the reason, offer alternatives
- For outages affecting work/business: flag as HIGH PRIORITY at the start of response
- Max 120 words per response
- End with: 'Is there anything else I can help with?'
"""

def optimized_chatbot(user_query):
    return prompt(user_query, system_msg=OPTIMIZED_SYSTEM_PROMPT, temperature=0.4, max_tokens=200)

print('✅ OPTIMIZED CHATBOT (with prompt engineering):')
print('=' * 60)
for q in test_queries:
    print(f'\n👤 {q}')
    print(f'🤖 {optimized_chatbot(q)}')
    print()

In [ ]:
# ── Step 3: Auto-Evaluate with LLM-as-Judge ──
# Use another LLM call to score the quality of responses

JUDGE_PROMPT = """
You are a quality evaluator for customer support responses.
Score the following response on these 4 dimensions (each 1-5):

1. Empathy (1=cold, 5=genuinely warm)
2. Helpfulness (1=vague, 5=specific actionable steps)
3. Professionalism (1=informal, 5=highly professional)
4. Conciseness (1=too long/short, 5=perfectly sized)

Return ONLY this JSON:
{{"empathy": X, "helpfulness": X, "professionalism": X, "conciseness": X, "overall": X, "feedback": "one line"}}

Customer query: {query}
Bot response: {response}
"""

import json

print('📊 LLM-as-Judge Evaluation')
print('=' * 60)
print(f'{"Query":<45} {"Naive":>6} {"Opt":>6} {"Diff":>6}')
print('-' * 65)

for q in test_queries:
    naive_resp = naive_chatbot(q)
    opt_resp   = optimized_chatbot(q)
    
    def score(response):
        p = JUDGE_PROMPT.format(query=q, response=response)
        result = prompt(p, temperature=0.0, max_tokens=120)
        try:
            return json.loads(result)
        except:
            return {'overall': 0}
    
    naive_score = score(naive_resp)['overall']
    opt_score   = score(opt_resp)['overall']
    diff = opt_score - naive_score
    diff_str = f'+{diff:.1f}' if diff >= 0 else str(diff)
    
    print(f'{q[:44]:<45} {naive_score:>6.1f} {opt_score:>6.1f} {diff_str:>6}')

print('\n(Scores: 1=poor to 5=excellent)')

In [ ]:
# ── ✏️  Your Turn: Improve the Prompt ──
# Modify the system prompt below and see if you can beat the optimized version

YOUR_SYSTEM_PROMPT = """
Write your own system prompt here.
Try to improve on the optimized version above.
Experiment with:
- Different personas
- Different response structures  
- Adding few-shot examples inside the system prompt
- Chain-of-thought instructions
"""

challenge_query = "I've been overcharged for 3 months in a row. I want to cancel and get a full refund!"

print('🏆 Challenge: Handle this angry customer better than the optimized bot!')
print('=' * 60)
print(f'\n👤 Customer: {challenge_query}\n')
print(f'✅ Optimized bot:')
print(optimized_chatbot(challenge_query))
print(f'\n🎯 Your bot:')
print(prompt(challenge_query, system_msg=YOUR_SYSTEM_PROMPT, temperature=0.4))

## ✅ Session 3 Summary

| Technique | Best For | Key Trick |
|---|---|---|
| **Zero-shot** | Classification, extraction | Be explicit about format |
| **Few-shot** | Pattern matching, custom formats | Use 3-5 diverse examples |
| **Chain-of-Thought** | Math, multi-step reasoning | "Think step by step" |
| **Role prompting** | Domain expertise, tone | Specific role, not generic |
| **Instructional** | Automation, pipelines | Exact schema + "return ONLY..." |
| **Negative prompting** | Avoiding bad outputs | Explicit DO NOT list |
| **Templates** | Production systems | Variables + validation |

## 🔗 What's Next
**Session 4** introduces **Vector Databases and RAG** — we will build a system that retrieves relevant context from a dataset and injects it into LLM prompts for accurate, grounded answers.

---
*Agentic AI Nano Bootcamp | Day 1 Session 3*